# Complex stft crn training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook trains the complex-STFT CRN used as the main STFT-domain comparison.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:
# =========================
# Imports and Drive
# =========================
import os, math, random, time, csv, json, warnings
from pathlib import Path
from typing import List, Optional, Dict, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not running in Colab or Drive already mounted:', repr(e))

try:
    import torchaudio
    import torchaudio.functional as AF
    HAVE_TORCHAUDIO = True
except Exception as e:
    HAVE_TORCHAUDIO = False
    print('torchaudio unavailable, will try librosa/soundfile fallback:', repr(e))

try:
    import soundfile as sf
    HAVE_SF = True
except Exception as e:
    HAVE_SF = False
    print('soundfile unavailable:', repr(e))

try:
    import librosa
    HAVE_LIBROSA = True
except Exception as e:
    HAVE_LIBROSA = False
    print('librosa unavailable:', repr(e))

from IPython.display import Audio, display, Markdown

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
# =========================
# Config
# =========================
SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DRIVE_ROOT = Path('/content/drive/MyDrive/master')
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"
CHECKPOINTS_RUNS_DIR = DRIVE_ROOT / 'checkpoints_runs'

# This is a new architecture run, not a fine-tune of the old plain STFT U-Net.
# It is DCCRN/CRN-inspired: convolutional encoder/decoder + recurrent temporal bottleneck.
INIT_CKPT = None
INIT_CKPT_PREFIX = None
REQUIRE_INIT_CKPT = False

RUN_NAME = "complex_stft_crn"
RUN_TAG = time.strftime('%Y%m%d_%H%M%S')
CKPT_DIR = CHECKPOINTS_RUNS_DIR / f'{RUN_NAME}_{RUN_TAG}'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV = CKPT_DIR / 'train_log.csv'

# Audio and data
SR = 22050
SEG_S = 2.0
SEG_LEN = int(SR * SEG_S)
BATCH = 3        # set to 2 if CUDA memory is tight; model has a BiGRU bottleneck
NUM_WORKERS = 2
PIN_MEMORY = (device.type == 'cuda')
P_SPEECH = 0.70

# STFT model grid. Use center=True to avoid PyTorch ISTFT overlap-add errors.
N_FFT = 1024
HOP = 256
WIN = 1024
CENTER = True
F_BINS = N_FFT // 2 + 1
BOT_FREQ_BINS = F_BINS // 8   # after three AvgPool2d(2) operations: 513 -> 256 -> 128 -> 64

# Inpainting setup
K_CHOICES = (1, 2)

# Training schedule
STEPS = 40000
LOG_EVERY = 100
SAVE_EVERY = 2500
LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

# Architecture
G_BASE = 16       # smaller than the old U-Net base because the recurrent bottleneck adds parameters
G_GROUPS = 8
G_IN_CH = 3       # real, imag, mask
G_OUT_CH = 2      # delta real, delta imag
CRN_HIDDEN = 256
CRN_LAYERS = 1
CRN_DROPOUT = 0.10

# Loss weights. Keep close to the original STFT model, with only modest mel regularization.
# Strong mel regularization caused buzzing in the v2 fine-tune, so this run uses mel losses only lightly.
LAMBDA_COMPLEX = 1.0
LAMBDA_LOGMAG = 20.0
LAMBDA_PHASE = 5.0
LAMBDA_WAV_MRSTFT = 3.0
LAMBDA_MEL = 2.0
LAMBDA_MEL_LOWMID = 1.0
LAMBDA_DELTA = 0.02

# High-frequency emphasis for STFT-domain losses
HF_START_FRAC = 0.45
HF_END_GAIN = 6.0
HF_RAMP_POWER = 2.0
MASK_DILATE = 2
PHASE_MASK_DILATE = 2
MEL_MASK_DILATE = 2

# Log-mel loss frontend. This is a training regularizer, not BigVGAN.
N_MELS = 80
MEL_FMIN = 0.0
MEL_FMAX = 8000.0
MEL_EPS = 1e-5
MEL_LOWMID_FRAC = 0.65

# Multi-resolution waveform STFT loss
MRSTFT_FFTS = (512, 1024, 2048)
MRSTFT_HOPS = (128, 256, 512)
MRSTFT_WINS = (512, 1024, 2048)

# Exact listening case used in earlier comparisons
EXACT_LISTEN_SPLIT_NAME = 'speech_test.txt'
EXACT_LISTEN_SPLIT_INDEX = 4
EXACT_LISTEN_K = 1
EXACT_LISTEN_OFFSET = 0
EXACT_LISTEN_SEG_S = 6.0

RUN_SMOKE_TEST_BEFORE_TRAINING = True

print('CKPT_DIR:', CKPT_DIR)
print('LOG_CSV:', LOG_CSV)
print('device:', device)
for p in [DRIVE_ROOT, SPLIT_DIR, CHECKPOINTS_RUNS_DIR]:
    print(p, 'exists:', p.exists())


In [ ]:
# =========================
# Data utilities
# =========================
def read_split_file(split_name: str) -> List[str]:
    path = SPLIT_DIR / split_name
    if not path.exists():
        raise FileNotFoundError(f'Missing split file: {path}')
    lines = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#'):
            lines.append(line)
    return lines


def resolve_audio_path(line: str) -> Path:
    raw = Path(line)
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    candidates += [
        DRIVE_ROOT / line,
        DRIVE_ROOT / 'datasets_large' / line,
        Path('/content') / line,
        Path(line),
    ]
    for c in candidates:
        if c.exists():
            return c
    # Return first candidate for a helpful error message later.
    return candidates[0]


def load_audio_file(path: Path, target_sr: int = SR) -> torch.Tensor:
    path = Path(path)
    if HAVE_TORCHAUDIO:
        wav, sr = torchaudio.load(str(path))  # [C,T]
        wav = wav.float()
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != target_sr:
            wav = AF.resample(wav, sr, target_sr)
        wav = wav.squeeze(0)
    elif HAVE_SF:
        arr, sr = sf.read(str(path), always_2d=True)
        arr = arr.mean(axis=1).astype(np.float32)
        if sr != target_sr:
            if not HAVE_LIBROSA:
                raise RuntimeError('Need librosa for resampling when torchaudio is unavailable.')
            arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
        wav = torch.from_numpy(arr).float()
    else:
        raise RuntimeError('No audio loader available. Install torchaudio or soundfile.')

    wav = torch.nan_to_num(wav)
    maxabs = wav.abs().max().clamp_min(1e-8)
    if maxabs > 1.0:
        wav = wav / maxabs
    return wav.clamp(-1.0, 1.0)


def random_crop_or_pad(wav: torch.Tensor, length: int) -> torch.Tensor:
    T = wav.numel()
    if T >= length:
        start = random.randint(0, T - length)
        return wav[start:start + length]
    out = torch.zeros(length, dtype=wav.dtype)
    out[:T] = wav
    return out


class AudioCropDataset(Dataset):
    def __init__(self, split_name: str, seg_len: int = SEG_LEN):
        self.lines = read_split_file(split_name)
        self.paths = [resolve_audio_path(x) for x in self.lines]
        self.seg_len = seg_len
        if len(self.paths) == 0:
            raise RuntimeError(f'No files in split {split_name}')
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        path = self.paths[idx % len(self.paths)]
        wav = load_audio_file(path, SR)
        wav = random_crop_or_pad(wav, self.seg_len)
        return wav

speech_train = AudioCropDataset('speech_train.txt', SEG_LEN)
music_train = AudioCropDataset('music_train.txt', SEG_LEN)
print('speech train:', len(speech_train), 'music train:', len(music_train))

speech_loader = DataLoader(speech_train, batch_size=BATCH, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
music_loader = DataLoader(music_train, batch_size=BATCH, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

speech_iter = iter(speech_loader)
music_iter = iter(music_loader)

def next_from(loader_name: str):
    global speech_iter, music_iter
    if loader_name == 'speech':
        try:
            return next(speech_iter)
        except StopIteration:
            speech_iter = iter(speech_loader)
            return next(speech_iter)
    else:
        try:
            return next(music_iter)
        except StopIteration:
            music_iter = iter(music_loader)
            return next(music_iter)

def next_mixed_batch():
    which = 'speech' if random.random() < P_SPEECH else 'music'
    wav = next_from(which).to(device, non_blocking=True)
    return wav, which

In [ ]:
# =========================
# DSP helpers: STFT, ISTFT, mel, masks, losses
# =========================
def hann_window(win: int, device=None):
    return torch.hann_window(win, device=device or globals().get('device', 'cpu'))


def stft_complex(wav_bt: torch.Tensor, n_fft=N_FFT, hop=HOP, win=WIN, center=CENTER):
    return torch.stft(
        wav_bt,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        window=hann_window(win, wav_bt.device),
        center=center,
        return_complex=True,
    )


def istft_complex(X_bft: torch.Tensor, length: int, n_fft=N_FFT, hop=HOP, win=WIN, center=CENTER):
    return torch.istft(
        X_bft,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        window=hann_window(win, X_bft.device),
        center=center,
        length=length,
    )


def match_frames_tensor(x: torch.Tensor, target_T: int):
    T = x.shape[-1]
    if T == target_T:
        return x
    if T > target_T:
        return x[..., :target_T]
    pad = target_T - T
    return F.pad(x, (0, pad))


def trim_complex_to_common(*Xs):
    T = min(x.shape[-1] for x in Xs)
    return tuple(x[..., :T] for x in Xs)


def dilate_mask_time(mask_b1t: torch.Tensor, radius: int):
    if radius <= 0:
        return mask_b1t.float()
    return F.max_pool1d(mask_b1t.float(), kernel_size=2 * radius + 1, stride=1, padding=radius)


def mask_to_ft(mask_b1t: torch.Tensor, n_freq: int):
    return mask_b1t.expand(-1, n_freq, -1)


def build_freq_weights(n_bins: int, start_frac: float, end_gain: float, power: float, device=None):
    idx = torch.linspace(0.0, 1.0, steps=n_bins, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    w = 1.0 + (end_gain - 1.0) * ramp
    return w.view(1, n_bins, 1)

STFT_FREQ_WEIGHTS = build_freq_weights(F_BINS, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)


def linear_time_inpaint_complex(X_bft: torch.Tensor, k: int, offset: int):
    # X_bft: [B,F,T] complex. Missing frames are filled between anchor frames.
    B, Fq, T = X_bft.shape
    X_interp = X_bft.clone()
    mask = torch.zeros(B, 1, T, device=X_bft.device, dtype=torch.float32)

    anchors = torch.zeros(T, device=X_bft.device, dtype=torch.bool)
    offset = int(offset) % (k + 1)
    anchors[offset::(k + 1)] = True
    anchor_idx = torch.where(anchors)[0]
    if anchor_idx.numel() < 2:
        return X_interp, mask

    for a, b in zip(anchor_idx[:-1].tolist(), anchor_idx[1:].tolist()):
        gap = b - a - 1
        if gap <= 0:
            continue
        left = X_bft[:, :, a]
        right = X_bft[:, :, b]
        for j in range(1, gap + 1):
            t = a + j
            alpha = j / (gap + 1)
            X_interp[:, :, t] = (1.0 - alpha) * left + alpha * right
            mask[:, :, t] = 1.0
    return X_interp, mask


def make_inpainting_pair_complex(X_real: torch.Tensor, k_choices=K_CHOICES, randomize_offset=True):
    k = int(random.choice(k_choices))
    offset = random.randint(0, k) if randomize_offset else 0
    X_interp, mask = linear_time_inpaint_complex(X_real, k=k, offset=offset)
    return dict(X_real=X_real, X_interp=X_interp, mask=mask, k=k, offset=offset)


def pack_complex_input(X_interp: torch.Tensor, mask: torch.Tensor):
    xr = X_interp.real.unsqueeze(1)
    xi = X_interp.imag.unsqueeze(1)
    xm = mask[:, :, None, :].expand(-1, 1, X_interp.shape[1], -1)
    return torch.cat([xr, xi, xm], dim=1)


def masked_complex_l1(X_hat, X_real, mask, freq_weights=None, mask_dilate=0):
    mask = dilate_mask_time(mask, mask_dilate)
    mask_ft = mask_to_ft(mask, X_hat.shape[1])
    w = 1.0 if freq_weights is None else freq_weights[..., :X_hat.shape[-1]]
    err = (X_hat.real - X_real.real).abs() + (X_hat.imag - X_real.imag).abs()
    denom = (mask_ft * w).sum().clamp_min(1e-8)
    return (err * mask_ft * w).sum() / denom


def masked_logmag_l1(X_hat, X_real, mask, freq_weights=None, mask_dilate=0):
    mask = dilate_mask_time(mask, mask_dilate)
    mask_ft = mask_to_ft(mask, X_hat.shape[1])
    w = 1.0 if freq_weights is None else freq_weights[..., :X_hat.shape[-1]]
    a = torch.log1p(X_hat.abs())
    b = torch.log1p(X_real.abs())
    denom = (mask_ft * w).sum().clamp_min(1e-8)
    return ((a - b).abs() * mask_ft * w).sum() / denom


def masked_phase_cos_loss(X_hat, X_real, mask, freq_weights=None, mask_dilate=0, eps=1e-8):
    mask = dilate_mask_time(mask, mask_dilate)
    mask_ft = mask_to_ft(mask, X_hat.shape[1])
    w = 1.0 if freq_weights is None else freq_weights[..., :X_hat.shape[-1]]
    ph_hat = X_hat / X_hat.abs().clamp_min(eps)
    ph_real = X_real / X_real.abs().clamp_min(eps)
    cos = (ph_hat * ph_real.conj()).real
    loss = 1.0 - cos
    denom = (mask_ft * w).sum().clamp_min(1e-8)
    return (loss * mask_ft * w).sum() / denom


def mrstft_loss(wav_hat: torch.Tensor, wav_real: torch.Tensor):
    total = 0.0
    for n_fft, hop, win in zip(MRSTFT_FFTS, MRSTFT_HOPS, MRSTFT_WINS):
        Xh = stft_complex(wav_hat, n_fft=n_fft, hop=hop, win=win, center=True)
        Xr = stft_complex(wav_real, n_fft=n_fft, hop=hop, win=win, center=True)
        Xh, Xr = trim_complex_to_common(Xh, Xr)
        mag_h = Xh.abs()
        mag_r = Xr.abs()
        log_l1 = (torch.log1p(mag_h) - torch.log1p(mag_r)).abs().mean()
        sc = (mag_h - mag_r).norm(p='fro') / mag_r.norm(p='fro').clamp_min(1e-8)
        total = total + log_l1 + 0.1 * sc
    return total / len(MRSTFT_FFTS)

# Mel filterbank without relying on librosa/torchaudio internals.
def hz_to_mel(f):
    return 2595.0 * torch.log10(torch.tensor(1.0, device=device) + f / 700.0)

def mel_to_hz(m):
    return 700.0 * (10.0 ** (m / 2595.0) - 1.0)

def build_mel_filterbank(sr: int, n_fft: int, n_mels: int, fmin: float, fmax: float, device=None):
    device = device or globals().get('device', 'cpu')
    fmin_t = torch.tensor(float(fmin), device=device)
    fmax_t = torch.tensor(float(fmax), device=device)
    m_min = hz_to_mel(fmin_t)
    m_max = hz_to_mel(fmax_t)
    m_pts = torch.linspace(m_min, m_max, n_mels + 2, device=device)
    f_pts = mel_to_hz(m_pts)
    bins = torch.floor((n_fft + 1) * f_pts / sr).long().clamp(0, n_fft // 2)
    fb = torch.zeros(n_fft // 2 + 1, n_mels, device=device)
    for m in range(1, n_mels + 1):
        left, center, right = bins[m - 1].item(), bins[m].item(), bins[m + 1].item()
        if center > left:
            fb[left:center, m - 1] = torch.linspace(0, 1, center - left, device=device)
        if right > center:
            fb[center:right, m - 1] = torch.linspace(1, 0, right - center, device=device)
    return fb.clamp_min(0.0)

MEL_FB = build_mel_filterbank(SR, N_FFT, N_MELS, MEL_FMIN, MEL_FMAX, device=device)
MEL_WEIGHTS_LOWMID = torch.ones(1, N_MELS, 1, device=device)
lowmid_end = int(N_MELS * MEL_LOWMID_FRAC)
MEL_WEIGHTS_LOWMID[:, lowmid_end:, :] = 0.25


def logmel_from_wav(wav_bt: torch.Tensor):
    X = stft_complex(wav_bt, n_fft=N_FFT, hop=HOP, win=WIN, center=CENTER)
    power = X.abs().pow(2)  # [B,F,T]
    mel = torch.matmul(power.transpose(1, 2), MEL_FB).transpose(1, 2)  # [B,M,T]
    return torch.log(mel.clamp_min(MEL_EPS))


def masked_mel_l1(mel_hat, mel_real, mask, weights=None, mask_dilate=0):
    T = min(mel_hat.shape[-1], mel_real.shape[-1], mask.shape[-1])
    mel_hat = mel_hat[..., :T]
    mel_real = mel_real[..., :T]
    mask = mask[..., :T]
    mask = dilate_mask_time(mask, mask_dilate)
    mask_mel = mask.expand(-1, mel_hat.shape[1], -1)
    w = 1.0 if weights is None else weights[..., :T]
    denom = (mask_mel * w).sum().clamp_min(1e-8)
    return ((mel_hat - mel_real).abs() * mask_mel * w).sum() / denom

In [ ]:
# =========================
# Model: Complex STFT CRN U-Net / DCCRN-lite inspired architecture
# =========================
def _valid_groups(ch: int, groups: int):
    groups = int(groups)
    while groups > 1 and ch % groups != 0:
        groups -= 1
    return max(1, groups)

class ComplexConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class TemporalBiGRUBottleneck(nn.Module):
    """BiGRU over the time axis at the U-Net bottleneck.

    Input is [B, C, F, T]. We flatten C and F, run a recurrent layer over T,
    then project back to the original bottleneck shape.

    This is not a full complex-valued DCCRN implementation, but it tests the
    same useful architectural idea: convolutional time-frequency features plus
    explicit temporal sequence modeling.
    """
    def __init__(self, channels: int, freq_bins: int, hidden: int = 256, layers: int = 1, dropout: float = 0.1):
        super().__init__()
        self.channels = channels
        self.freq_bins = freq_bins
        self.input_size = channels * freq_bins
        self.gru = nn.GRU(
            input_size=self.input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.proj = nn.Linear(2 * hidden, self.input_size)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(self.input_size)

    def forward(self, x):
        B, C, Fq, T = x.shape
        if Fq != self.freq_bins:
            raise RuntimeError(f'Temporal bottleneck expected freq_bins={self.freq_bins}, got {Fq}. Check N_FFT/pooling.')
        seq = x.permute(0, 3, 1, 2).contiguous().view(B, T, C * Fq)
        y, _ = self.gru(seq)
        y = self.proj(y)
        y = self.norm(seq + self.drop(y))
        y = y.view(B, T, C, Fq).permute(0, 2, 3, 1).contiguous()
        return y

class ComplexSTFTCRNUNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=16, groups=8, bottleneck_freq_bins=64,
                 crn_hidden=256, crn_layers=1, crn_dropout=0.1):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.temporal = TemporalBiGRUBottleneck(
            channels=base*8,
            freq_bins=bottleneck_freq_bins,
            hidden=crn_hidden,
            layers=crn_layers,
            dropout=crn_dropout,
        )
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)

        # Start close to the interpolation baseline: zero-ish residual at initialization.
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))
        xb = self.temporal(xb)

        y3 = F.interpolate(xb, size=x3.shape[-2:], mode='bilinear', align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

G = ComplexSTFTCRNUNet2D(
    in_ch=G_IN_CH,
    out_ch=G_OUT_CH,
    base=G_BASE,
    groups=G_GROUPS,
    bottleneck_freq_bins=BOT_FREQ_BINS,
    crn_hidden=CRN_HIDDEN,
    crn_layers=CRN_LAYERS,
    crn_dropout=CRN_DROPOUT,
).to(device)
print('G params:', sum(p.numel() for p in G.parameters()) / 1e6, 'M')


In [ ]:
# =========================
# Checkpoint loading and saving
# =========================
def find_latest_checkpoint(prefix: str, root: Path = CHECKPOINTS_RUNS_DIR) -> Optional[Path]:
    if prefix is None:
        return None
    root = Path(root)
    dirs = sorted([p for p in root.glob(prefix + '*') if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    for d in dirs:
        latest = d / 'latest.pt'
        if latest.exists():
            return latest
        pts = sorted(d.glob('*.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
        if pts:
            return pts[0]
    return None


def safe_torch_load(path: Path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for key in ['G', 'model', 'generator', 'state_dict', 'model_state_dict']:
            if key in ckpt and isinstance(ckpt[key], dict):
                sd = ckpt[key]
                break
        else:
            sd = ckpt
    else:
        raise RuntimeError('Checkpoint is not a dict.')
    clean = {}
    for k, v in sd.items():
        kk = k
        for pref in ['module.', 'G.', 'model.', 'generator.']:
            if kk.startswith(pref):
                kk = kk[len(pref):]
        clean[kk] = v
    return clean

init_path = Path(INIT_CKPT) if INIT_CKPT is not None else find_latest_checkpoint(INIT_CKPT_PREFIX)
print('INIT checkpoint:', init_path)
if init_path is None:
    if REQUIRE_INIT_CKPT:
        raise FileNotFoundError(f'No init checkpoint found for prefix {INIT_CKPT_PREFIX}. Set INIT_CKPT manually.')
    print('Training new CRN U-Net from scratch.')
else:
    ckpt = safe_torch_load(init_path, map_location=device)
    sd = extract_state_dict(ckpt)
    missing, unexpected = G.load_state_dict(sd, strict=False)
    print('Loaded init checkpoint:', init_path)
    print('missing keys:', len(missing), missing[:10])
    print('unexpected keys:', len(unexpected), unexpected[:10])

opt_g = torch.optim.AdamW(G.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.8, 0.99))

def save_ckpt(step: int, name: Optional[str] = None):
    payload = dict(
        step=step,
        run_name=RUN_NAME,
        model_kind='ComplexSTFTCRNUNet2D',
        config=dict(
            SR=SR, N_FFT=N_FFT, HOP=HOP, WIN=WIN, CENTER=CENTER,
            G_BASE=G_BASE, G_GROUPS=G_GROUPS,
            CRN_HIDDEN=CRN_HIDDEN, CRN_LAYERS=CRN_LAYERS, CRN_DROPOUT=CRN_DROPOUT,
            BOT_FREQ_BINS=BOT_FREQ_BINS,
            init_ckpt=str(init_path) if init_path is not None else None,
            losses=dict(
                LAMBDA_COMPLEX=LAMBDA_COMPLEX, LAMBDA_LOGMAG=LAMBDA_LOGMAG,
                LAMBDA_PHASE=LAMBDA_PHASE, LAMBDA_WAV_MRSTFT=LAMBDA_WAV_MRSTFT,
                LAMBDA_MEL=LAMBDA_MEL, LAMBDA_MEL_LOWMID=LAMBDA_MEL_LOWMID,
                LAMBDA_DELTA=LAMBDA_DELTA,
            ),
        ),
        G=G.state_dict(),
        opt_g=opt_g.state_dict(),
    )
    path = CKPT_DIR / (name or f'A1CPLXCRN_step{step:07d}.pt')
    torch.save(payload, path)
    torch.save(payload, CKPT_DIR / 'latest.pt')
    return path


In [ ]:
# =========================
# Forward + loss computation
# =========================
def forward_batch(wav: torch.Tensor, randomize_offset=True):
    wav = wav.clamp(-1.0, 1.0)
    X_real = stft_complex(wav)
    pair = make_inpainting_pair_complex(X_real, k_choices=K_CHOICES, randomize_offset=randomize_offset)
    X_interp = pair['X_interp']
    mask = pair['mask']
    k = pair['k']; offset = pair['offset']

    x_in = pack_complex_input(X_interp, mask)
    delta = G(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask_to_ft(mask, X_interp.shape[1])
    X_hat = X_interp + dX * mask_ft

    wav_hat = istft_complex(X_hat, length=wav.shape[-1])
    wav_hat = torch.nan_to_num(wav_hat).clamp(-1.2, 1.2)

    # STFT-domain losses
    loss_complex = masked_complex_l1(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_logmag = masked_logmag_l1(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_phase = masked_phase_cos_loss(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=PHASE_MASK_DILATE)
    loss_delta = (dX.abs() * mask_ft).sum() / mask_ft.sum().clamp_min(1e-8)

    # Waveform / mel clarity losses
    loss_wav_mrstft = mrstft_loss(wav_hat, wav)
    mel_hat = logmel_from_wav(wav_hat)
    mel_real = logmel_from_wav(wav)
    mel_mask = match_frames_tensor(mask, mel_hat.shape[-1])
    loss_mel = masked_mel_l1(mel_hat, mel_real, mel_mask, weights=None, mask_dilate=MEL_MASK_DILATE)
    loss_mel_lowmid = masked_mel_l1(mel_hat, mel_real, mel_mask, weights=MEL_WEIGHTS_LOWMID, mask_dilate=MEL_MASK_DILATE)

    loss_total = (
        LAMBDA_COMPLEX * loss_complex +
        LAMBDA_LOGMAG * loss_logmag +
        LAMBDA_PHASE * loss_phase +
        LAMBDA_WAV_MRSTFT * loss_wav_mrstft +
        LAMBDA_MEL * loss_mel +
        LAMBDA_MEL_LOWMID * loss_mel_lowmid +
        LAMBDA_DELTA * loss_delta
    )

    with torch.no_grad():
        base_logmag = masked_logmag_l1(X_interp, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
        ref_logmag = masked_logmag_l1(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
        base_complex = masked_complex_l1(X_interp, X_real, mask, freq_weights=None, mask_dilate=MASK_DILATE)
        ref_complex = masked_complex_l1(X_hat, X_real, mask, freq_weights=None, mask_dilate=MASK_DILATE)

    return dict(
        wav=wav, wav_hat=wav_hat,
        X_real=X_real, X_interp=X_interp, X_hat=X_hat,
        mask=mask, delta=dX, k=k, offset=offset,
        loss_total=loss_total,
        loss_complex=loss_complex,
        loss_logmag=loss_logmag,
        loss_phase=loss_phase,
        loss_wav_mrstft=loss_wav_mrstft,
        loss_mel=loss_mel,
        loss_mel_lowmid=loss_mel_lowmid,
        loss_delta=loss_delta,
        base_logmag=base_logmag,
        ref_logmag=ref_logmag,
        base_complex=base_complex,
        ref_complex=ref_complex,
    )

In [ ]:
# =========================
# Smoke test before long training
# =========================
if RUN_SMOKE_TEST_BEFORE_TRAINING:
    print('Running smoke test...')
    G.train()
    wav, which = next_mixed_batch()
    wav = wav[:1]
    opt_g.zero_grad(set_to_none=True)
    out = forward_batch(wav)
    out['loss_total'].backward()
    grad_norm = torch.sqrt(sum((p.grad.detach().float().norm() ** 2) for p in G.parameters() if p.grad is not None))
    opt_g.zero_grad(set_to_none=True)

    print('batch source:', which)
    print('loss_total:', float(out['loss_total'].detach().cpu()))
    print('loss_logmag:', float(out['loss_logmag'].detach().cpu()))
    print('loss_mel:', float(out['loss_mel'].detach().cpu()))
    print('grad_norm:', float(grad_norm.detach().cpu()))
    print('X shapes real/interp/hat:', tuple(out['X_real'].shape), tuple(out['X_interp'].shape), tuple(out['X_hat'].shape))
    print('wav_hat shape:', tuple(out['wav_hat'].shape))
    assert torch.isfinite(out['loss_total']).item(), 'Smoke-test loss is not finite.'
    assert grad_norm > 0, 'No gradient reached the model.'
    print('Smoke test passed.')

In [ ]:
# =========================
# Training loop
# =========================
fieldnames = [
    'step', 'source', 'k', 'offset',
    'loss_total', 'loss_complex', 'loss_logmag', 'loss_phase', 'loss_wav_mrstft',
    'loss_mel', 'loss_mel_lowmid', 'loss_delta',
    'base_logmag', 'ref_logmag', 'base_complex', 'ref_complex',
]

with open(LOG_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

start_time = time.time()
G.train()
for step in range(1, STEPS + 1):
    wav, source = next_mixed_batch()

    opt_g.zero_grad(set_to_none=True)
    out = forward_batch(wav)
    loss = out['loss_total']
    loss.backward()
    torch.nn.utils.clip_grad_norm_(G.parameters(), GRAD_CLIP)
    opt_g.step()

    if step % LOG_EVERY == 0 or step == 1:
        row = dict(
            step=step,
            source=source,
            k=out['k'],
            offset=out['offset'],
        )
        for key in fieldnames:
            if key in row:
                continue
            if key in out:
                row[key] = float(out[key].detach().cpu())
        with open(LOG_CSV, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writerow(row)

        elapsed = (time.time() - start_time) / 60
        print(
            f"step {step:7d} | {source:6s} | loss {row['loss_total']:.4f} | "
            f"logmag {row['loss_logmag']:.4f} | mel {row['loss_mel']:.4f} | "
            f"mrstft {row['loss_wav_mrstft']:.4f} | base/ref logmag {row['base_logmag']:.4f}/{row['ref_logmag']:.4f} | "
            f"k {out['k']} off {out['offset']} | {elapsed:.1f} min"
        )

    if step % SAVE_EVERY == 0:
        path = save_ckpt(step)
        print('saved:', path)

final_path = save_ckpt(STEPS, name=f'A1CPLXCRN_step{STEPS:07d}.pt')
print('done. final:', final_path)
print('CKPT_DIR:', CKPT_DIR)

In [ ]:
# =========================
# Plot training curves
# =========================
import pandas as pd
import matplotlib.pyplot as plt

if LOG_CSV.exists():
    df = pd.read_csv(LOG_CSV)
    display(df.tail())
    for cols in [
        ['loss_total'],
        ['loss_logmag', 'base_logmag', 'ref_logmag'],
        ['loss_mel', 'loss_mel_lowmid'],
        ['loss_complex', 'loss_phase', 'loss_wav_mrstft'],
    ]:
        plt.figure(figsize=(9, 4))
        for c in cols:
            if c in df.columns:
                plt.plot(df['step'], df[c], label=c)
        plt.xlabel('step')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print('No log found:', LOG_CSV)

In [ ]:
# =========================
# Exact listening case after / during training
# =========================
def read_eval_clip(split_name: str, index: int, seg_s: float):
    lines = read_split_file(split_name)
    line = lines[index]
    path = resolve_audio_path(line)
    wav = load_audio_file(path, SR)
    target_len = int(seg_s * SR)
    # Deterministic start: use beginning if too short, else first target_len samples.
    if wav.numel() < target_len:
        wav = random_crop_or_pad(wav, target_len)
    else:
        wav = wav[:target_len]
    return path, wav.unsqueeze(0).to(device)

@torch.no_grad()
def run_exact_case(ckpt_path: Optional[Path] = None):
    if ckpt_path is not None:
        ck = safe_torch_load(Path(ckpt_path), map_location=device)
        G.load_state_dict(extract_state_dict(ck), strict=False)
    G.eval()
    path, wav = read_eval_clip(EXACT_LISTEN_SPLIT_NAME, EXACT_LISTEN_SPLIT_INDEX, EXACT_LISTEN_SEG_S)
    X_real = stft_complex(wav)
    X_interp, mask = linear_time_inpaint_complex(X_real, k=EXACT_LISTEN_K, offset=EXACT_LISTEN_OFFSET)
    x_in = pack_complex_input(X_interp, mask)
    delta = G(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask_to_ft(mask, X_interp.shape[1])
    X_hat = X_interp + dX * mask_ft
    wav_interp = istft_complex(X_interp, length=wav.shape[-1]).clamp(-1,1)
    wav_hat = istft_complex(X_hat, length=wav.shape[-1]).clamp(-1,1)

    display(Markdown(f'### Exact case: `{path}`'))
    display(Markdown(f'split={EXACT_LISTEN_SPLIT_NAME}, index={EXACT_LISTEN_SPLIT_INDEX}, k={EXACT_LISTEN_K}, offset={EXACT_LISTEN_OFFSET}'))
    display(Markdown('**Original waveform**'))
    display(Audio(wav[0].detach().cpu().numpy(), rate=SR))
    display(Markdown('**Interpolated complex STFT → ISTFT**'))
    display(Audio(wav_interp[0].detach().cpu().numpy(), rate=SR))
    display(Markdown('**Refined complex STFT CRN U-Net → ISTFT**'))
    display(Audio(wav_hat[0].detach().cpu().numpy(), rate=SR))

    metrics = dict(
        base_logmag=float(masked_logmag_l1(X_interp, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).cpu()),
        ref_logmag=float(masked_logmag_l1(X_hat, X_real, mask, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).cpu()),
        base_complex=float(masked_complex_l1(X_interp, X_real, mask, mask_dilate=MASK_DILATE).cpu()),
        ref_complex=float(masked_complex_l1(X_hat, X_real, mask, mask_dilate=MASK_DILATE).cpu()),
        ref_mel=float(masked_mel_l1(logmel_from_wav(wav_hat), logmel_from_wav(wav), match_frames_tensor(mask, logmel_from_wav(wav_hat).shape[-1]), mask_dilate=MEL_MASK_DILATE).cpu()),
    )
    display(pd.DataFrame([metrics]))
    return dict(wav=wav, wav_interp=wav_interp, wav_hat=wav_hat, X_real=X_real, X_interp=X_interp, X_hat=X_hat, mask=mask, metrics=metrics)

# Run with latest checkpoint from this run.
_ = run_exact_case(CKPT_DIR / 'latest.pt')